In [1]:
import arcgis
import time
from arcgis.gis import GIS
from arcgis.gis import Item
from arcgis.apps.storymap import StoryMap

from typing import Set  # Import Set from typing
import re, json, csv
from collections import OrderedDict

import pandas as pd
import os
import logging
import requests

# Set Pandas dataframe display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns',1000)

In [2]:
agoNotebook = False
# Print the version of the arcgis module
print(f"Running ArcGIS API for Python version: {arcgis.__version__}")

# Define the GIS
if agoNotebook == False:
    import keyring
    service_name = "system" # Use the default local credential store
    success = False # Set initial state

    # Ask for the username
    while success == False:
        username_for_keyring = input("Enter your ArcGIS Online username:") # If you are using VS Code, the text input dialog box appears at the top of the window
        # Get the credential object
        credential = keyring.get_credential(service_name, username_for_keyring)
        # Check if the username is in the credential store
        if credential is None:
            print(f"'{username_for_keyring}' is not in the local system's credential store. Try another username.")
        # Retrieve the password, login and set the GIS portal
        else:
            password_from_keyring = keyring.get_password("system", username_for_keyring)
            portal_url = 'https://www.arcgis.com'  
            gis = GIS(portal_url, username=username_for_keyring, password=password_from_keyring)
            success = True
            # Print a success message with username and user's organization role
            print("Successfully logged in as: " + gis.properties.user.username, "(role: " + gis.properties.user.role + ")")
else:
    gis = GIS("home")

Running ArcGIS API for Python version: 2.4.2
Successfully logged in as: dasbury_storymaps (role: org_admin)


In [3]:
classic_maptour_id = "d79e17055aa14e119c9c6e8621b23a6a" # "73f4483f851b4f8f92eb4efedaf98957" #Esri Campus Tour
# classic_maptour_webmap = ""
# classic_maptour_featureCollection = ""
# classic_maptour_featureSet = ""

In [4]:
from converter_json import *

# Retrieve the JSON data for the classic item
classic_item = gis.content.get(classic_maptour_id)
classic_json = classic_item.get_data()


In [5]:
target_story_id, new_storymap_json = convert_classic_to_json(classic_json, theme_id="summit", gis_token=gis._con.token, gis=gis)

Fetching json from webmap item
Found Map Tour layer
Found webmap_json in classic_json
Uploaded resource: place_000_img.jpg
Uploaded resource: place_001_img.jpg
Uploaded resource: place_002_img.jpg
Uploaded resource: place_003_img.jpg
Uploaded resource: place_004_img.jpg
Uploaded resource: place_005_img.jpg
Uploaded resource: place_006_img.jpg
Uploaded resource: place_007_img.jpg
Uploaded resource: place_008_img.jpg
Uploaded resource: place_009_img.jpg
Uploaded resource: place_010_img.jpg
Uploaded resource: place_011_img.jpg
Uploaded resource: place_012_img.jpg
Uploaded resource: place_013_img.jpg
Uploaded resource: place_014_img.jpg
Uploaded resource: place_015_img.jpg
Uploaded resource: place_016_img.jpg
Uploaded resource: place_017_img.jpg
Uploaded resource: place_018_img.jpg
Uploaded resource: place_019_img.jpg
Uploaded resource: place_020_img.jpg
Uploaded resource: place_021_img.jpg
Uploaded resource: place_022_img.jpg
Webmap ID: ee03727216e64a0b82d6a2b624ad09ed
Found webmap_json i

In [6]:
print(target_story_id)

e98e9eaad8084d229cc9e2e0ad302267


In [7]:
# reorder nodes to match builder output
def get_node_type_order(reference_json_path):
    with open(reference_json_path, "r", encoding="utf-8") as f:
        ref_json = json.load(f)
    type_order = []
    for node_id in ref_json["nodes"]:
        node_type = ref_json["nodes"][node_id].get("type")
        if node_type and node_type not in type_order:
            type_order.append(node_type)
    return type_order

def order_nodes_for_output(storymap_json, type_order):
    # Group nodes by type
    nodes_by_type = {t: [] for t in type_order}
    other_nodes = []
    for node_id, node in storymap_json["nodes"].items():
        node_type = node.get("type")
        if node_type in nodes_by_type:
            nodes_by_type[node_type].append((node_id, node))
        else:
            other_nodes.append((node_id, node))

    # Build ordered dict by type order
    ordered_nodes = OrderedDict()
    for t in type_order:
        for node_id, node in nodes_by_type[t]:
            ordered_nodes[node_id] = node
    # Add any nodes with types not in type_order at the end
    for node_id, node in other_nodes:
        ordered_nodes[node_id] = node

    storymap_json["nodes"] = ordered_nodes
    return storymap_json

In [8]:
type_order = get_node_type_order("map-tour-duplicated.json")
new_storymap_json = order_nodes_for_output(new_storymap_json, type_order)

In [9]:
with open("map-tour-converted.json", "w", encoding="utf-8") as f:
    json.dump(new_storymap_json, f, indent=4, ensure_ascii=False)

In [10]:
# Assume you have:
username = gis.properties.user.username
token = gis._con.token

# 1. Find the draft resource name (e.g., "draft_*.json")
resources_url = f"https://www.arcgis.com/sharing/rest/content/items/{target_story_id}/resources?f=json&token={token}"
resources_response = requests.get(resources_url).json()
for res in resources_response.get("resources", []):
    resource_name = res.get("resource", "")
    if resource_name.startswith("draft_") and resource_name.endswith(".json"):
        draft_resource_name = resource_name
        break
if not draft_resource_name:
    raise Exception("Could not find draft resource in target StoryMap.")

# 2. Remove the old draft resource
remove_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{target_story_id}/removeResources"
remove_params = {
    "f": "json",
    "token": token,
    "resource": draft_resource_name
}
remove_response = requests.post(remove_resource_url, data=remove_params)
assert remove_response.json().get("success"), "Failed to remove old draft resource"

# 3. Upload the new draft resource
add_resource_url = f"https://www.arcgis.com/sharing/rest/content/users/{username}/items/{target_story_id}/addResources"
json_blob = json.dumps(new_storymap_json)
files = {"file": (draft_resource_name, json_blob)}
add_params = {
    "f": "json",
    "token": token,
    "fileName": draft_resource_name
}
add_response = requests.post(add_resource_url, files=files, data=add_params)
assert add_response.json().get("success"), "Failed to upload new draft resource"

print(f"Updated {draft_resource_name} in StoryMap {target_story_id}")

Updated draft_1762483462298.json in StoryMap e98e9eaad8084d229cc9e2e0ad302267
